In [1]:
%pip install selenium beautifulsoup4 requests webdriver-manager

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.5 MB 12.6 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 41.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import time
import random
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

def download_image_and_create_label(img_url, img_dir, label_dir, img_name):
    try:
        # Thêm User-Agent vào Request tải ảnh để không bị máy chủ ảnh chặn
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
        }
        response = requests.get(img_url, headers=headers, stream=True, timeout=10)
        
        if response.status_code == 200:
            img_path = os.path.join(img_dir, f"{img_name}.jpg")
            with open(img_path, 'wb') as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
            
            txt_path = os.path.join(label_dir, f"{img_name}.txt")
            with open(txt_path, 'w') as f:
                pass
                
            return True
    except Exception as e:
        pass
    return False

def crawl_hard_negatives(target_count=4000):
    base_dir = "0_normal_dataset"
    img_dir = os.path.join(base_dir, "images")
    label_dir = os.path.join(base_dir, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(label_dir, exist_ok=True)
    
    print(f"📁 Dữ liệu sẽ lưu tại: {base_dir}/")

    queries = [
        "restaurant kitchen steam",
        "industrial welding smoke",
        "welding sparks factory",
        "factory steam pipe",
        "dusty woodworking shop",
        "indoor vape smoke cloud",
        "dry ice fog event",
        "incense smoke indoor",
        "factory halogen warning light",
        "red indicator light machine"
    ]

    chrome_options = Options()
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    
    total_downloaded = 0
    pages_per_query = 15 

    for query in queries:
        if total_downloaded >= target_count: break
            
        print(f"\n{'='*50}")
        print(f"🔍 ĐANG QUÉT TỪ KHÓA: '{query}'")
        print(f"{'='*50}")
        
        query_formatted = query.replace(' ', '-')
        query_prefix = query.replace(' ', '_')
        
        for page in range(1, pages_per_query + 1):
            if total_downloaded >= target_count: break
                
            url = f"https://www.alamy.com/stock-photo/{query_formatted}.html?page={page}&sortBy=relevant"
            driver.get(url)
            time.sleep(random.uniform(2.5, 4.0)) 
            
            # --- BƯỚC TỐI ƯU 1: CUỘN TRANG TỪ TỪ (SMOOTH SCROLL) ---
            # Thay vì cuộn thẳng xuống đáy, ta cuộn từng đoạn 800 pixel để ép mọi ảnh phải load
            scroll_pause_time = 0.8
            screen_height = driver.execute_script("return window.screen.height;")
            i = 1
            while True:
                # Cuộn xuống 1 đoạn bằng chiều cao màn hình
                driver.execute_script(f"window.scrollTo(0, {screen_height} * {i});")  
                i += 1
                time.sleep(scroll_pause_time)
                
                scroll_height = driver.execute_script("return document.body.scrollHeight;")  
                # Nếu đã cuộn đến đáy trang thì dừng
                if (screen_height * i) > scroll_height:
                    break

            # Lấy source HTML sau khi mọi thứ đã được ép load
            soup = BeautifulSoup(driver.page_source, "html.parser")
            images = soup.find_all("img")
            
            page_count = 0
            for img in images:
                if total_downloaded >= target_count: break
                
                # --- BƯỚC TỐI ƯU 2: BẮT LINK ẢNH TRIỆT ĐỂ ---
                # Nhiều trang web giấu link thật ở data-src hoặc srcset
                src = img.get("data-src") or img.get("src")
                
                if src and "alamy" in src.lower() and (".jpg" in src.lower() or ".png" in src.lower()):
                    # Tách bỏ phần rác sau đuôi file (vd: .jpg?v=123)
                    src = src.split('?')[0]
                    
                    if src.startswith("//"):
                        src = "https:" + src
                        
                    img_name = f"normal_{query_prefix}_p{page}_{page_count:03d}"
                    
                    if download_image_and_create_label(src, img_dir, label_dir, img_name):
                        page_count += 1
                        total_downloaded += 1
                        
                        if total_downloaded % 50 == 0:
                            print(f"  [+] Tiến độ: {total_downloaded}/{target_count} ảnh")

            print(f"👉 Xong Trang {page} ('{query}'): Lấy được {page_count} ảnh.")
            
            # Đã nới lỏng điều kiện ngắt vòng lặp (Chỉ bỏ qua nếu lấy được dưới 3 ảnh)
            if page_count < 3:
                print(f"⚠️ Trang web có thể đã chặn hoặc cạn ảnh. Chuyển từ khóa tiếp theo!")
                break

    driver.quit()
    print(f"\n🎉 HOÀN TẤT NHIỆM VỤ! Tổng số ảnh Hard Negatives đã tải: {total_downloaded}")

if __name__ == "__main__":
    crawl_hard_negatives(target_count=4000)

📁 Dữ liệu sẽ lưu tại: 0_normal_dataset/

🔍 ĐANG QUÉT TỪ KHÓA: 'restaurant kitchen steam'
  [+] Tiến độ: 50/4000 ảnh
  [+] Tiến độ: 100/4000 ảnh
👉 Xong Trang 1 ('restaurant kitchen steam'): Lấy được 100 ảnh.
  [+] Tiến độ: 150/4000 ảnh
  [+] Tiến độ: 200/4000 ảnh
👉 Xong Trang 2 ('restaurant kitchen steam'): Lấy được 100 ảnh.
  [+] Tiến độ: 250/4000 ảnh
  [+] Tiến độ: 300/4000 ảnh
👉 Xong Trang 3 ('restaurant kitchen steam'): Lấy được 100 ảnh.
  [+] Tiến độ: 350/4000 ảnh
  [+] Tiến độ: 400/4000 ảnh
👉 Xong Trang 4 ('restaurant kitchen steam'): Lấy được 100 ảnh.
  [+] Tiến độ: 450/4000 ảnh
  [+] Tiến độ: 500/4000 ảnh
👉 Xong Trang 5 ('restaurant kitchen steam'): Lấy được 100 ảnh.
  [+] Tiến độ: 550/4000 ảnh
  [+] Tiến độ: 600/4000 ảnh
👉 Xong Trang 6 ('restaurant kitchen steam'): Lấy được 100 ảnh.
  [+] Tiến độ: 650/4000 ảnh
  [+] Tiến độ: 700/4000 ảnh
👉 Xong Trang 7 ('restaurant kitchen steam'): Lấy được 100 ảnh.
  [+] Tiến độ: 750/4000 ảnh
  [+] Tiến độ: 800/4000 ảnh
👉 Xong Trang 8 ('restau